# Préparation des données `Véhicules` — BAAC 2020–2024

Ce notebook est consacré à la préparation des données relatives aux véhicules impliqués dans les accidents corporels de la circulation pour la période 2020–2024.

L'objectif est de vérifier la compatibilité des fichiers annuels, d'identifier les variables pertinentes pour la problématique, de consolider les données, d'évaluer leur qualité puis d'appliquer les traitements nécessaires afin d'obtenir un jeu de données nettoyé et exploitable pour les prochaines étapes du projet.

## 1. Préparation et consolidation des données

### 1.1. Chargement des fichiers annuels

Les fichiers `Véhicules` des années 2020 à 2024 sont chargés séparément afin de pouvoir comparer leur structure avant leur consolidation.

In [359]:
# 1.1. Chargement des fichiers annuels

import pandas as pd

df_vehicules_2020 = pd.read_csv(
    "data/raw/2020/vehicules-2020.csv",
    sep=";"
)

df_vehicules_2021 = pd.read_csv(
    "data/raw/2021/vehicules-2021.csv",
    sep=";"
)

df_vehicules_2022 = pd.read_csv(
    "data/raw/2022/vehicules-2022.csv",
    sep=";"
)

df_vehicules_2023 = pd.read_csv(
    "data/raw/2023/vehicules-2023.csv",
    sep=";"
)

df_vehicules_2024 = pd.read_csv(
    "data/raw/2024/vehicules-2024.csv",
    sep=";"
)

### 1.2. Vérification de la structure et de la compatibilité

La structure des fichiers annuels est comparée afin d'identifier les éventuelles différences de colonnes ou de types avant leur consolidation.

Cette vérification permettra également d'examiner le rôle des variables et leur pertinence pour la problématique afin d'orienter leur sélection dès la phase de préparation.

In [360]:
# 1.2. Vérification de la structure et de la compatibilité

dfs_vehicules = {
    2020: df_vehicules_2020,
    2021: df_vehicules_2021,
    2022: df_vehicules_2022,
    2023: df_vehicules_2023,
    2024: df_vehicules_2024
}

for annee, dataframe in dfs_vehicules.items():
    print(f"{annee} : {dataframe.shape}")
    print(dataframe.columns.tolist())
    print(dataframe.dtypes)
    print()

2020 : (81066, 11)
['Num_Acc', 'id_vehicule', 'num_veh', 'senc', 'catv', 'obs', 'obsm', 'choc', 'manv', 'motor', 'occutc']
Num_Acc          int64
id_vehicule        str
num_veh            str
senc             int64
catv             int64
obs              int64
obsm             int64
choc             int64
manv             int64
motor            int64
occutc         float64
dtype: object

2021 : (97315, 11)
['Num_Acc', 'id_vehicule', 'num_veh', 'senc', 'catv', 'obs', 'obsm', 'choc', 'manv', 'motor', 'occutc']
Num_Acc          int64
id_vehicule        str
num_veh            str
senc             int64
catv             int64
obs              int64
obsm             int64
choc             int64
manv             int64
motor            int64
occutc         float64
dtype: object

2022 : (94493, 11)
['Num_Acc', 'id_vehicule', 'num_veh', 'senc', 'catv', 'obs', 'obsm', 'choc', 'manv', 'motor', 'occutc']
Num_Acc          int64
id_vehicule        str
num_veh            str
senc             int64
cat

#### Résultat

Les fichiers `Véhicules` 2020 à 2024 présentent une structure homogène : ils contiennent les mêmes **11 variables**, dans le même ordre, avec des types techniques identiques.

Aucune harmonisation des noms de colonnes n'est nécessaire avant la consolidation.

Une première analyse métier permet également d'identifier les variables directement pertinentes pour la problématique et celles dont l'utilité doit être vérifiée avant leur sélection définitive. Les variables `num_veh`, `senc` et `occutc` feront notamment l'objet d'une vérification ciblée afin d'éviter de conserver des informations redondantes ou peu exploitables.

### 1.3. Vérification des identifiants et de la granularité

Les identifiants sont vérifiés avant la consolidation afin de confirmer la granularité de la table `Véhicules` et d'évaluer le rôle de `Num_Acc`, `id_vehicule` et `num_veh`.

Une même valeur de `Num_Acc` peut normalement apparaître plusieurs fois puisqu'un accident peut impliquer plusieurs véhicules.

In [361]:
# 1.3. Vérification des identifiants et de la granularité

for annee, dataframe in dfs_vehicules.items():
    print(
        f"{annee} : "
        f"{len(dataframe)} lignes | "
        f"{dataframe['Num_Acc'].nunique()} accidents | "
        f"{dataframe['id_vehicule'].nunique()} véhicules"
    )

2020 : 81066 lignes | 47744 accidents | 81066 véhicules
2021 : 97315 lignes | 56518 accidents | 97315 véhicules
2022 : 94493 lignes | 55302 accidents | 94493 véhicules
2023 : 93585 lignes | 54822 accidents | 93585 véhicules
2024 : 92678 lignes | 54402 accidents | 92678 véhicules


In [362]:
# 1.3. Vérification de l'identifiant num_veh

for annee, dataframe in dfs_vehicules.items():
    doublons = dataframe.duplicated(
        subset=["Num_Acc", "num_veh"]
    ).sum()

    print(f"{annee} : {doublons} doublons (Num_Acc, num_veh)")

2020 : 0 doublons (Num_Acc, num_veh)
2021 : 0 doublons (Num_Acc, num_veh)
2022 : 2 doublons (Num_Acc, num_veh)
2023 : 0 doublons (Num_Acc, num_veh)
2024 : 1 doublons (Num_Acc, num_veh)


#### Résultat

Une ligne de la table `Véhicules` correspond à un véhicule impliqué dans un accident.

`Num_Acc` peut donc apparaître plusieurs fois lorsqu'un accident implique plusieurs véhicules. À l'inverse, `id_vehicule` est unique dans chacun des fichiers annuels et constitue l'identifiant technique le plus fiable du véhicule.

Le couple (`Num_Acc`, `num_veh`) présente quelques doublons en 2022 et 2024 alors que les `id_vehicule` concernés sont distincts. La variable `num_veh`, redondante avec `id_vehicule` pour notre exploitation et moins fiable comme identifiant, ne sera donc pas retenue dans la sélection finale.

`id_vehicule` est conservée pour identifier les véhicules et permettre notamment la liaison avec la table `Usagers`.

### 1.4. Vérification de la variable `occutc`

La variable `occutc` indique le nombre d'occupants d'un véhicule de transport en commun. Son champ d'application étant très spécifique, sa disponibilité est vérifiée avant de décider de son maintien dans l'analyse.

In [363]:
# 1.4. Vérification de la variable occutc

for annee, dataframe in dfs_vehicules.items():
    nb_manquantes = dataframe["occutc"].isna().sum()
    taux_manquantes = dataframe["occutc"].isna().mean() * 100

    print(
        f"{annee} : "
        f"{nb_manquantes} valeurs manquantes "
        f"({taux_manquantes:.2f} %)"
    )

2020 : 80445 valeurs manquantes (99.23 %)
2021 : 96571 valeurs manquantes (99.24 %)
2022 : 93676 valeurs manquantes (99.14 %)
2023 : 92747 valeurs manquantes (99.10 %)
2024 : 91729 valeurs manquantes (98.98 %)


#### Résultat

La variable `occutc` présente environ 99 % de valeurs manquantes sur chacune des années 2020 à 2024. Ce taux s'explique par son champ d'application spécifique aux véhicules de transport en commun.

Compte tenu de sa faible disponibilité et de son intérêt limité pour la problématique étudiée, `occutc` ne sera pas retenue dans la sélection finale.

La sélection des variables conduit également à écarter `num_veh`, redondante avec `id_vehicule` pour notre exploitation, ainsi que `senc`, dont l'intérêt analytique est limité pour l'étude des facteurs associés à la gravité des accidents.

Les 8 variables retenues sont : `Num_Acc`, `id_vehicule`, `catv`, `obs`, `obsm`, `choc`, `manv` et `motor`.

### 1.5. Sélection des variables et concaténation

Les variables retenues sont sélectionnées avant la concaténation afin de ne conserver que les informations utiles à l'analyse.

Les fichiers annuels 2020 à 2024 ayant une structure compatible, ils sont ensuite regroupés dans un DataFrame unique.

In [364]:
# 1.5. Sélection des variables et concaténation

colonnes_vehicules = [
    "Num_Acc",
    "id_vehicule",
    "catv",
    "obs",
    "obsm",
    "choc",
    "manv",
    "motor"
]

df_vehicules_concat = pd.concat(
    [
        df_vehicules_2020[colonnes_vehicules],
        df_vehicules_2021[colonnes_vehicules],
        df_vehicules_2022[colonnes_vehicules],
        df_vehicules_2023[colonnes_vehicules],
        df_vehicules_2024[colonnes_vehicules]
    ],
    ignore_index=True
)

df_vehicules_concat.shape

(459137, 8)

### 1.6. Sauvegarde des données consolidées

La version consolidée des données `Véhicules` est enregistrée dans le dossier `interim`. Elle servira de base aux contrôles de qualité et aux traitements de nettoyage.

In [365]:
# 1.6. Sauvegarde des données consolidées

df_vehicules_concat.to_csv(
    "data/interim/vehicules_2020_2024.csv",
    sep=";",
    index=False
)

## 2. Évaluation de la qualité des données

La qualité des données consolidées est évaluée avant leur nettoyage afin d'identifier les éventuels doublons, valeurs manquantes ou modalités nécessitant un traitement.

Les contrôles sont ciblés sur les huit variables retenues et sont interprétés en tenant compte de leur signification dans la documentation BAAC.

In [366]:
# 2.1. Contrôle des doublons

nb_doublons = df_vehicules_concat.duplicated().sum()

print(f"Nombre de doublons complets : {nb_doublons}")

Nombre de doublons complets : 0


#### Résultat

Aucun doublon complet n'est détecté dans les données consolidées. Aucune suppression de doublons n'est donc nécessaire.

In [367]:
# 2.2. Contrôle des valeurs manquantes

valeurs_manquantes = pd.DataFrame({
    "Nombre": df_vehicules_concat.isna().sum(),
    "Pourcentage": (df_vehicules_concat.isna().mean() * 100).round(2)
})

valeurs_manquantes

,Nombre,Pourcentage
Num_Acc,0,0.0
id_vehicule,0,0.0
catv,0,0.0
obs,0,0.0
obsm,0,0.0
choc,0,0.0
manv,0,0.0
motor,0,0.0


#### Résultat

Aucune valeur manquante au format `NaN` n'est détectée dans les huit variables retenues.

Cependant, certaines variables BAAC utilisent des codes spécifiques pour représenter une information non renseignée ou une situation particulière. Un contrôle des modalités est donc nécessaire avant de conclure sur la complétude réelle des données.

In [368]:
# 2.3. Contrôle des modalités

variables_vehicules = [
    "catv",
    "obs",
    "obsm",
    "choc",
    "manv",
    "motor"
]

for variable in variables_vehicules:
    print(f"\n{variable} :")
    print(df_vehicules_concat[variable].value_counts().sort_index())


catv :
catv
-1         17
 0       1267
 1      26117
 2      16762
 3       2392
 7     268557
 10     32900
 13      1825
 14      3409
 15      4552
 16       167
 17      2904
 20       417
 21      1236
 30     12991
 31      8184
 32      9781
 33     36666
 34      4474
 35       116
 36       857
 37      3154
 38       815
 39       159
 40       597
 41        85
 42       102
 43      2712
 50      9742
 60      1118
 80      3083
 99      1979
Name: count, dtype: int64

obs :
obs
-1        207
 0     391405
 1      10900
 2       7040
 3       6006
 4       6466
 5        882
 6       5309
 7       1378
 8       5458
 9       2324
 10       501
 11       770
 12      3815
 13      8171
 14      3352
 15      2654
 16      2026
 17       473
Name: count, dtype: int64

obsm :
obsm
-1       201
 0     86093
 1     41702
 2    321423
 4       484
 5       409
 6       640
 9      8185
Name: count, dtype: int64

choc :
choc
-1       254
 0     29851
 1    166403
 2     55704
 3

#### Résultat

Les modalités observées sont globalement cohérentes avec les codes définis dans la documentation BAAC.

Les six variables descriptives contiennent quelques valeurs `-1`, correspondant à des informations non renseignées. Elles seront converties en valeurs manquantes lors du nettoyage.

Les valeurs `0` ne sont pas traitées globalement comme des valeurs manquantes, car leur signification dépend de chaque variable dans le dictionnaire BAAC. Elles sont donc conservées lorsqu'elles correspondent à une modalité définie.

In [369]:
# Vérification des modalités non prévues

modalites_valides = {
    "catv": {-1, 0, 1, 2, 3, 7, 10, 13, 14, 15, 16, 17,
             20, 21, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39,
             40, 41, 42, 43, 50, 60, 80, 99},

    "obs": {-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12,
            13, 14, 15, 16, 17},

    "obsm": {-1, 0, 1, 2, 4, 5, 6, 9},

    "choc": {-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9},

    "manv": {-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,
             14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26},

    "motor": {-1, 0, 1, 2, 3, 4, 5, 6}
}

for variable, valeurs_valides in modalites_valides.items():
    valeurs_observees = set(df_vehicules_concat[variable].unique())
    valeurs_inattendues = valeurs_observees - valeurs_valides

    print(f"{variable} : {sorted(valeurs_inattendues)}")

catv : []
obs : []
obsm : []
choc : []
manv : []
motor : []


#### Résultat

Aucune modalité inattendue n'est détectée pour les variables `catv`, `obs`, `obsm`, `choc`, `manv` et `motor`.

Les valeurs observées sont donc cohérentes avec les modalités prévues dans la documentation BAAC. Les valeurs `-1`, correspondant à des informations non renseignées, seront traitées lors du nettoyage.

### 2.4. Synthèse des décisions de nettoyage

Les contrôles de structure, de pertinence métier et de qualité permettent de définir les traitements à appliquer avant la création de la version nettoyée.

| Élément | Constat | Décision |
|---|---|---|
| `Num_Acc` | Identifiant de l'accident, répété normalement lorsqu'un accident implique plusieurs véhicules | Conserver |
| `id_vehicule` | Identifiant unique du véhicule dans chaque fichier annuel et nécessaire pour la liaison avec `Usagers` | Conserver |
| `num_veh` | Identifiant secondaire, redondant avec `id_vehicule` pour notre exploitation et présentant quelques doublons au sein d'un même accident | Supprimer |
| `senc` | Sens de circulation, peu pertinent pour l'analyse des facteurs associés à la gravité | Supprimer |
| `occutc` | Variable spécifique aux transports en commun et environ 99 % de valeurs manquantes | Supprimer |
| `catv` | Catégorie du véhicule, pertinente pour l'analyse de la gravité | Conserver |
| `obs` | Obstacle fixe heurté, pertinent pour caractériser les circonstances de l'accident | Conserver |
| `obsm` | Obstacle mobile heurté, pertinent pour caractériser le type de collision | Conserver |
| `choc` | Point de choc initial, pertinent pour caractériser la configuration du choc | Conserver |
| `manv` | Manœuvre principale avant l'accident, pertinente pour l'analyse des circonstances | Conserver |
| `motor` | Type de motorisation, potentiellement utile pour l'analyse | Conserver |
| Doublons complets | Aucun doublon complet détecté après sélection des variables | Aucun traitement |
| Valeurs `NaN` | Aucune valeur `NaN` initialement dans les 8 variables retenues | Aucun traitement direct |
| Codes `-1` | Présents dans les 6 variables descriptives et correspondant à des informations non renseignées | Remplacer par `pd.NA` |
| Codes `0` | Modalités dont la signification dépend de la variable dans le dictionnaire BAAC | Conserver |
| Modalités inattendues | Aucune modalité hors des codes attendus détectée | Aucun traitement |

## 3. Nettoyage des données

Les traitements définis lors de l'évaluation de la qualité sont appliqués afin d'obtenir une version nettoyée et exploitable de la table `Véhicules`.

Les variables non retenues ont déjà été écartées avant la consolidation. Le nettoyage porte donc principalement sur le traitement des codes correspondant à des informations non renseignées et sur l'harmonisation des types de données.

Une validation finale sera ensuite réalisée avant l'export des données nettoyées.

### 3.1. Création du DataFrame de nettoyage

Une copie des données consolidées est créée afin d'appliquer les traitements de nettoyage sans modifier la version intermédiaire de référence.

In [370]:
# 3.1. Création du DataFrame de nettoyage

df_vehicules_clean = df_vehicules_concat.copy()

### 3.2. Traitement des valeurs non renseignées

Les codes `-1`, correspondant à des informations non renseignées dans les variables descriptives retenues, sont remplacés par des valeurs manquantes (`pd.NA`).

Les valeurs `0` sont conservées conformément à leur signification dans le dictionnaire BAAC.

In [371]:
# 3.2. Traitement des valeurs non renseignées

variables_a_nettoyer = [
    "catv",
    "obs",
    "obsm",
    "choc",
    "manv",
    "motor"
]

df_vehicules_clean[variables_a_nettoyer] = (
    df_vehicules_clean[variables_a_nettoyer]
    .replace(-1, pd.NA)
)

In [372]:
#  Vérification du traitement

for variable in variables_a_nettoyer:
    nb_moins_un = (df_vehicules_clean[variable] == -1).sum()
    print(f"{variable} : {nb_moins_un}")

catv : 0
obs : 0
obsm : 0
choc : 0
manv : 0
motor : 0


### 3.3. Harmonisation des types

Les variables descriptives sont converties en type entier nullable (`Int64`) afin de conserver des valeurs numériques tout en permettant la présence de valeurs manquantes (`pd.NA`).

Les identifiants sont conservés dans un format adapté aux futures jointures avec les autres tables.

In [373]:
# 3.3. Harmonisation des types

df_vehicules_clean["Num_Acc"] = df_vehicules_clean["Num_Acc"].astype("string")
df_vehicules_clean["id_vehicule"] = df_vehicules_clean["id_vehicule"].astype("string")

for variable in variables_a_nettoyer:
    df_vehicules_clean[variable] = df_vehicules_clean[variable].astype("Int64")

In [374]:
# Vérification des types

df_vehicules_clean.dtypes

Num_Acc        string
id_vehicule    string
catv            Int64
obs             Int64
obsm            Int64
choc            Int64
manv            Int64
motor           Int64
dtype: object

### 3.4. Renommage des variables

Les variables descriptives sont renommées avec des intitulés explicites afin de faciliter leur compréhension et leur utilisation dans les prochaines étapes du projet.

Les identifiants `Num_Acc` et `id_vehicule` conservent leur nom d'origine afin de maintenir des clés communes entre les différentes tables BAAC.

In [375]:
# 3.4. Renommage des variables

df_vehicules_clean = df_vehicules_clean.rename(
    columns={
        "catv": "categorie_vehicule",
        "obs": "obstacle_fixe",
        "obsm": "obstacle_mobile",
        "choc": "point_choc_initial",
        "manv": "manoeuvre_principale",
        "motor": "motorisation"
    }
)

df_vehicules_clean.columns.tolist()

['Num_Acc',
 'id_vehicule',
 'categorie_vehicule',
 'obstacle_fixe',
 'obstacle_mobile',
 'point_choc_initial',
 'manoeuvre_principale',
 'motorisation']

### 3.5. Validation finale

Une validation finale est réalisée afin de vérifier la cohérence du jeu de données après nettoyage et renommage des variables, avant son export.

In [376]:
# 3.5. Validation finale

print("Dimensions :", df_vehicules_clean.shape)
print("Accidents uniques :", df_vehicules_clean["Num_Acc"].nunique())
print("Véhicules uniques :", df_vehicules_clean["id_vehicule"].nunique())
print("Doublons complets :", df_vehicules_clean.duplicated().sum())
print("Num_Acc manquants :", df_vehicules_clean["Num_Acc"].isna().sum())
print("id_vehicule manquants :", df_vehicules_clean["id_vehicule"].isna().sum())

print("\nValeurs manquantes après nettoyage :")
print(df_vehicules_clean.isna().sum())

Dimensions : (459137, 8)
Accidents uniques : 268788
Véhicules uniques : 459137
Doublons complets : 0
Num_Acc manquants : 0
id_vehicule manquants : 0

Valeurs manquantes après nettoyage :
Num_Acc                   0
id_vehicule               0
categorie_vehicule       17
obstacle_fixe           207
obstacle_mobile         201
point_choc_initial      254
manoeuvre_principale    162
motorisation            912
dtype: int64


#### Résultat

La table nettoyée contient **459 137 véhicules** répartis dans **268 788 accidents** sur la période 2020–2024.

`id_vehicule` est unique sur l'ensemble des données consolidées et aucune valeur manquante n'est présente dans les identifiants `Num_Acc` et `id_vehicule`. Aucun doublon complet n'est détecté.

Les valeurs manquantes présentes dans les variables descriptives correspondent aux codes `-1` identifiés comme des informations non renseignées et volontairement convertis en valeurs manquantes.

La table `Véhicules` est donc prête à être exportée et utilisée dans les prochaines étapes du projet.

### 3.6. Export des données nettoyées

La version finale nettoyée de la table `Véhicules` est enregistrée dans le dossier `processed` afin d'être utilisée pour les prochaines étapes d'analyse, de modélisation et d'intégration des données.

In [377]:
# 3.6. Export des données nettoyées

df_vehicules_clean.to_csv(
    "data/processed/vehicules_2020_2024_clean.csv",
    sep=";",
    index=False
)